In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Conv1D, MaxPooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

def train_lstm_cnn(csv_path="data/asos_seoul_daily_enriched.csv", #LSTM+CNN 모델을 훈련하는 함수 정의
                   model_path="models/lstm_cnn_model.h5",
                   scaler_path="models/lstm_cnn_scaler.pkl"):
    df = pd.read_csv(csv_path) #모델링용 데이터 로드

    features = ['avgTa', 'minTa', 'maxTa', 'sumRn', 'avgWs', 'avgRhm', 'avgTs', 'avgTd', 'avgPs'] #모델에 사용할 9개 기상 특성
    target = 'flood_risk' #예측할 타겟변수(침수 위험도) 문자열로

    df = df.dropna(subset=features + [target]) #결측치 제거
    scaler = MinMaxScaler() #0~1 범위로 데이터를 정규화할 MinMaxScaler 객체 생성
    X_scaled = scaler.fit_transform(df[features]) #특성 데이터를 0~1 범위로 정규화하고 스케일러에 변환 규칙 학습
    y = df[target].values #타겟 데이터를 numpy 배열로 변환

    def create_sequences(X, y, window_size=7): #시계열 데이터를 7일 윈도우 시퀀스로 변환하는 내부 함수
        X_seq, y_seq = [], [] #시퀀스 데이터와 타겟을 저장
        for i in range(len(X) - window_size): #전체 데이터 길이에서 윈도우 크기를 뺀 만큼 반복
            X_seq.append(X[i:i+window_size]) #i번째부터 7일간의 기상 데이터를 시퀀스에 추가
            y_seq.append(y[i+window_size]) #7일 후의 홍수 위험도를 타겟에 추가
        return np.array(X_seq), np.array(y_seq) #리스트를 numpy 배열로 변환하여 반환

    X_seq, y_seq = create_sequences(X_scaled, y) #시퀀스 데이터 생성

    X_train, X_test, y_train, y_test = train_test_split( #데이터 분할
        X_seq, y_seq, test_size=0.2, stratify=y_seq, random_state=42
    )

    # LSTM + CNN 모델 정의
    model = Sequential()
    model.add(Conv1D(32, kernel_size=2, activation='relu', input_shape=(X_seq.shape[1], X_seq.shape[2]))) #1D 합성곱 레이어 추가: 32개 필터, 커널 크기 2, ReLU 활성화, 입력 형태 (7, 9)
    model.add(MaxPooling1D(pool_size=2)) #최대 풀링 레이어 추가: 크기 2로 다운샘플링
    model.add(LSTM(64)) #LSTM 레이어 추가: 64개 히든 유닛
    model.add(Dropout(0.3))
    model.add(Dense(64, activation='relu')) #완전연결 레이어 추가: 64개 뉴런, ReLU 활성화
    model.add(Dense(1, activation='sigmoid')) #출력층:1개 뉴런, sigmoid 활성화

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy']) #모델 컴파일

    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True) #검증 손실을 모니터링하여 3 에포크 동안 개선이 없으면 조기 종료
    model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2, callbacks=[early_stop]) #훈련 데이터로 모델 학습: 최대 20 에포크, 배치 크기 32, 검증 데이터 20%, 조기 종료 콜백 사용

    #모델 예측 및 평가
    y_pred_prob = model.predict(X_test).ravel() #테스트 데이터에 대해 홍수 확률 예측하고 1차원 배열로 변환
    y_pred = (y_pred_prob > 0.5).astype(int) #확률값이 0.5 이상이면 True(1), 미만이면 False(0)로 변환

    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred)) #혼동행렬을 계산하고 콘솔에 출력 (실제값 vs 예측값의 교차표)
    print("\nClassification Report:\n", classification_report(y_test, y_pred)) #정밀도, 재현율, F1-점수 등의 분류 성능 지표들을 표 형태로 출력함
    print("\nROC AUC Score:", roc_auc_score(y_test, y_pred_prob)) #ROC 곡선 아래 면적을 계산하여 모델의 전체적인 분류 성능을 하나의 숫자로 출력함

    # 저장
    model.save(model_path)
    joblib.dump(scaler, scaler_path)
    print(f"모델 저장 완료: {model_path}")
    print(f"스케일러 저장 완료: {scaler_path}")

    # 시각화 저장
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob) #ROC 곡선을 위한 거짓양성률(FPR)과 참양성률(TPR)을 계산 (세 번째 값은 사용하지 않으므로 _로 무시)
    plt.figure()
    plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc_score(y_test, y_pred_prob):.3f}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("LSTM+CNN ROC Curve")
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.savefig("outputs/lstm_cnn_roc_curve.png")
    plt.show()

    prec, rec, _ = precision_recall_curve(y_test, y_pred_prob) #PR 곡선을 위한 정밀도와 재현율을 계산
    plt.figure()
    plt.plot(rec, prec)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("LSTM+CNN Precision-Recall Curve")
    plt.grid()
    plt.tight_layout()
    plt.savefig("outputs/lstm_cnn_precision_recall_curve.png")
    plt.show()